# Neural JSCC for point processes over a jitter channel — v2

Full rework of `hawkes_jscc.ipynb` based on the same assignment
(`student_project_point_process_jscc.pdf`): same Gaussian jitter channel, same
rate $R = T_s/T_c = 1$, same constraint $N_X = N_S$, same distortion (eq. 7-8).

## Why a v2: diagnosing v1

In v1, the constraint $N_X = N_S$ was **already correct** (top-N by rank and
`gather_at_spikes` guaranteed it 100% of the time). The real problem was elsewhere.

Comparing v1's numbers to a **blind decoder** — a decoder that completely
ignores $Y$ and outputs $\mathbb{E}[T_i \mid N, i]$ — gives:

| source | D blind | v1 `jscc` at $\sigma/\mu = 4$ | v1 `jscc` at $\sigma/\mu = 0.1$ |
|---|---|---|---|
| poisson | 0.0185 | 0.0305 | — |
| gamma k=4 | 0.0051 | 0.0130 | — |
| hawkes | 0.0341 | 0.0498 | 0.0498 |

v1's distortion was **flat** across four decades of noise and of the same
order as the blind decoder: the system was transmitting no temporal
information at all. The "4 to 15× gain over `uncoded`" at high noise was not a
JSCC gain, only the fact that a prior-based predictor beats raw transmission
when SNR is bad.

**Root cause: the identity was not representable.** With
$\Delta t_i = \mathrm{softplus}(\text{head}(u_i))$ and $u$ the potential of a
leaky LIF, the gap $g_i$ is encoded in $u \approx a\beta^{g_i/\Delta} + b$, i.e.
exponentially. Reproducing the input would require a logarithm, and `head` is
linear. The best function in that family is $\Delta t \approx$ constant —
exactly the solution found. This was not an optimization problem.

## The five changes

1. **Displacement parametrization** (Step 3 of the PDF, "encoder displacement
   constraint"): $g^X_i = g_i\,e^{a_i}$ and $\hat{S}_i = Y_{(i)} + c_i$, with the
   readout heads initialized to zero. **At initialization, the system *is*
   `uncoded`** — it can only improve, and the worry "the decoder should at
   least match uncoded" disappears by construction.
2. **Event-driven SNN**: the LIF is unrolled event by event, with the leak
   between two events being $e^{-g_i/\tau}$, the *exact* solution of the ODE.
   It's the same neuron as in discrete time, just without quantizing time.
   Consequences: no more bin collisions (v1 had 26% to 58% of them **in the
   source itself**), no more 128-step unrolling, and $N$ trivially preserved.
3. **Blind baseline everywhere**: every table and every figure carries the
   $D_{\text{blind}}$ line. That's the number that tells you whether the
   system is transmitting anything at all.
4. **No `min(N_S, N_Ŝ)` truncation**: the three counts are equal by
   construction, so the distortion is exactly eq. 7. In v1, normalizing by
   `min()` *rewarded* dropping events.
5. **Step 3 with the numerical optimum**: for the two-event model, we compute
   the optimal encoder (a free monotone function on a grid) and the exact
   MMSE decoder. We compare the network to the **optimum**, not to `uncoded`.

## What the notebook shows

- The JSCC gain exists and it is **large**: in the two-event model, the
  optimal encoder does 2 to 10× better than the best decoder alone.
- The mechanism is a **dilation**: the encoder spreads the distribution of
  gaps over the whole window $[0, T_c]$ (local slope $\approx 6.6$ on Gamma),
  which increases the effective SNR of the timing channel at constant event
  cost.
- In the full $N$-event problem, the gain is **much smaller** (10-20%) — and
  that's consistent: the source already fills the window
  ($\sum_i g_i \approx T_c$), so the encoder can only *reallocate*, not dilate.

**Full run time: on the order of 30 minutes on 4 CPU cores** (≈15 min
for experiment 1, 7 min for experiment 2, 6 min for Step 3). The `STEPS`,
`SEEDS`, and `SIGMA_GRID` constants in the configuration cell allow scaling
down: `STEPS = 200, SEEDS = [0]` runs the whole thing in about 5 minutes,
enough to check that the pipeline works (the gaps between `decoder` and
`jscc` then become too noisy to interpret).

> **Validation status.** The computational building blocks — sources,
> `EventLIF`, `ResidualSystem`, `distortion`, `train`, `evaluate`, `Grid`,
> `optimal_encoder`, `train_tiny` — have been run and verified. The figure
> and table cells have been reviewed but not executed: if one breaks, it's a
> display issue, not a computation issue.


## 1. Imports and configuration

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, replace
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(4)
DEVICE = "cpu"


@dataclass
class Config:
    # ------- source / channel -------------------------------------------------
    Ts: float = 1.0              # source block duration ; Tc = Ts so R = 1
    mean_gap: float = 1.0 / 9.0  # source time scale (µ)
    sigma_ratio: float = 1.0     # sigma_J / mean_gap  (the only noise parameter)

    # ------- network ----------------------------------------------------------
    hidden: int = 32             # width of the single LIF layer
    n_in: int = 2                # features per event: [g_i/µ, T_i/Tc]
    spike_alpha: float = 2.0     # arctan surrogate slope
    a_max: float = math.log(4.0) # max dilation of a gap by the encoder (x4 / /4)
    c_max_ratio: float = 3.0     # max decoder displacement, x max(µ, sigma_J)

    # ------- optimization ------------------------------------------------------
    steps: int = 600
    batch: int = 128
    lr: float = 3e-3
    seed: int = 0

    @property
    def Tc(self) -> float:
        return self.Ts                      # R = Ts/Tc = 1

    @property
    def sigma_J(self) -> float:
        return self.sigma_ratio * self.mean_gap


CFG = Config()

# --- run constants (reduce for a quick test) --------------------
STEPS       = 600
SEEDS       = [0, 1]
SIGMA_GRID  = [0.3, 1.0, 3.0]
N_TRAIN, N_EVAL = 20_000, 8_000
MODES = ("uncoded", "decoder", "jscc")

print(CFG)
print(f"sigma_J = {CFG.sigma_J:.4f}   (mean_gap = {CFG.mean_gap:.4f})")


### Plotting conventions

Same palette as the previous notebooks so all three read together.
`C_BLIND` (grey) is new: it's the blind-decoder line.


In [ ]:
C_UNCODED, C_DECODER, C_JSCC = "#2a78d6", "#eb6834", "#1baf7a"
C_BLIND, C_OPT = "#8a8983", "#b0342d"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8983", "#e4e3df", "#fcfcfb"
MODE_STYLE = {"uncoded": (C_UNCODED, "o"), "decoder": (C_DECODER, "s"),
              "jscc": (C_JSCC, "^")}


def style(ax, xlabel, ylabel, title, subtitle=None):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9, length=3, color=GRID)
    ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    ax.set_title(title, color=INK, fontsize=12, loc="left", pad=26 if subtitle else 8)
    if subtitle:
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes, color=MUTED,
                fontsize=9, va="bottom")


def legend(ax, loc="best"):
    leg = ax.legend(frameon=False, fontsize=9, loc=loc)
    for t in leg.get_texts():
        t.set_color(INK2)


def show_table(rows, cols=None, title=""):
    cols = cols or list(rows[0].keys())
    w = {c: max(len(c), 11) for c in cols}
    if title:
        print(title)
    print(" | ".join(c.rjust(w[c]) for c in cols))
    print("-+-".join("-" * w[c] for c in cols))
    for r in rows:
        cells = []
        for c in cols:
            v = r.get(c, "")
            if isinstance(v, float):
                v = f"{v:.4e}" if (v != 0 and abs(v) < 1e-3) else f"{v:.4f}"
            cells.append(str(v).rjust(w[c]))
        print(" | ".join(cells))

## 2. Sources — event-based representation

**Change of representation compared to v1.** A block is no longer a binary
raster of `T_bins` steps but a list of continuous times, right-padded and
accompanied by a mask: `(t (B, N_max), mask (B, N_max))`. There is no longer
any time quantization anywhere in the pipeline.

Why this matters: v1 measured `frac_source_collision` between 0.26 and
0.58 — in more than one block out of four, **the source itself** lost at
least one event during rasterization, before any encoding. The diagnostic
cell below re-quantifies this effect to justify the change.

Three families, matching the three steps of the PDF:

- **homogeneous Poisson** (Step 1): exponential renewal, the most "random"
  source;
- **Gamma renewal** (Step 2, eq. 10): $W_i \sim \Gamma(k, \mu/k)$, with fixed
  mean, $k$ swept over $\{1, 2, 4, 8\}$;
- **Hawkes** with exponential kernel (extension of v1): $\lambda(t) = \mu_0 +
  \alpha\sum_{T_i<t} e^{-\beta(t-T_i)}$, multi-event memory and bursts.

In all three cases the window $[0, T_s)$ is fixed and $N_S$ is **random**: it
is a consequence of the draw, never a parameter.


In [ ]:
def _pack(blocks):
    """List of time arrays -> (t (B,N_max), mask (B,N_max))."""
    N = max(max(len(b) for b in blocks), 1)
    t = np.zeros((len(blocks), N), dtype=np.float32)
    m = np.zeros((len(blocks), N), dtype=bool)
    for i, b in enumerate(blocks):
        t[i, :len(b)] = b
        m[i, :len(b)] = True
    return torch.from_numpy(t).to(DEVICE), torch.from_numpy(m).to(DEVICE)


def gen_renewal(cfg, n_blocks, seed, k=1.0):
    """Vectorized Gamma renewal, W_i ~ Gamma(k, µ/k). k = 1 -> Poisson."""
    rng = np.random.default_rng(seed)
    K = int(6 * cfg.Ts / cfg.mean_gap) + 20          # margin: P(N > K) negligible
    T = np.cumsum(rng.gamma(k, cfg.mean_gap / k, size=(n_blocks, K)), axis=1)
    keep = T < cfg.Ts
    return _pack([T[i, keep[i]] for i in range(n_blocks)])


def _hawkes_one(mu, alpha, beta, Ts, n_burn, cap, rng):
    """One block via Ogata's (1981) thinning, with a burn-in to start in the
    stationary regime. We stop as soon as the candidate time exceeds Ts."""
    t, e = 0.0, 0.0
    for _ in range(n_burn):
        M = mu + e
        tau = rng.exponential(1.0 / M); e *= np.exp(-beta * tau); t += tau
        if rng.random() <= (mu + e) / M:
            e += alpha
    t0, out = t, []
    while len(out) < cap:
        M = mu + e
        tau = rng.exponential(1.0 / M); e *= np.exp(-beta * tau); t += tau
        if t - t0 >= Ts:
            break
        if rng.random() <= (mu + e) / M:
            out.append(t - t0); e += alpha
    return np.array(out)


def gen_hawkes(cfg, n_blocks, seed, branching=0.6, decay=1.0, n_burn=40, cap=200):
    """n = alpha/beta in (0,1): share of the rate due to self-excitation.
    mu0, alpha, beta are derived so that the stationary rate stays 1/mean_gap."""
    rng = np.random.default_rng(seed)
    beta = 1.0 / (decay * cfg.mean_gap)
    alpha = branching * beta
    mu = (1.0 / cfg.mean_gap) * (1.0 - branching)
    return _pack([_hawkes_one(mu, alpha, beta, cfg.Ts, n_burn, cap, rng)
                  for _ in range(n_blocks)])


SOURCES = {
    "poisson":  lambda cfg, n, s: gen_renewal(cfg, n, s, k=1.0),
    "gamma_k2": lambda cfg, n, s: gen_renewal(cfg, n, s, k=2.0),
    "gamma_k4": lambda cfg, n, s: gen_renewal(cfg, n, s, k=4.0),
    "gamma_k8": lambda cfg, n, s: gen_renewal(cfg, n, s, k=8.0),
    "hawkes":   lambda cfg, n, s: gen_hawkes(cfg, n, s, branching=0.6),
}


class Dataset:
    """Pre-generated dataset. Generating blocks once and sampling from them
    avoids paying the cost of simulating the source at every training step
    (that was the main cost in v1)."""

    def __init__(self, t, m):
        self.t, self.m = t, m

    def sample(self, batch, gen):
        idx = torch.randint(0, self.t.shape[0], (batch,), generator=gen)
        t, m = self.t[idx], self.m[idx]
        N = int(m.sum(1).max().item())
        return t[:, :N].clone(), m[:, :N].clone()


def gaps(t, m):
    """(B,N) increasing times -> (B,N) gaps. g_0 = T_1 (from the origin),
    g_i = T_{i+1} - T_i afterwards: this is eq. 12 of the PDF shifted by one
    index."""
    prev = torch.cat([torch.zeros_like(t[:, :1]), t[:, :-1]], 1)
    return (t - prev) * m.to(t.dtype)


In [ ]:
# --- source statistics + the cost that rasterization would have had --------------
def source_stats(name, cfg, n=4000, seed=12345, T_bins=128):
    t, m = SOURCES[name](cfg, n, seed)
    n_s = m.sum(1).float()
    bw = cfg.Ts / T_bins
    idx = (t / bw).long()
    coll = 0
    tn, mn = idx.numpy(), m.numpy()
    for b in range(tn.shape[0]):
        v = tn[b][mn[b]]
        coll += int(len(np.unique(v)) != len(v))
    return {"source": name, "mean_n_s": float(n_s.mean()), "std_n_s": float(n_s.std()),
            "max_n_s": int(n_s.max()),
            f"frac_collision_if_T_bins_{T_bins}": coll / tn.shape[0]}


rows = [source_stats(s, CFG) for s in SOURCES]
show_table(rows, title="Source statistics (random N_S, fixed window)")
print()
print("The last column is the fraction of blocks that WOULD LOSE at least one event")
print("if rasterized at T_bins=128, as v1 did. This is the reason for the")
print("switch to an event-based representation: here, this rate is 0 by construction.")


In [ ]:
def fig_sources(cfg, names=("poisson", "gamma_k8", "hawkes"), n_blocks=12, seed=3):
    fig, axes = plt.subplots(len(names), 1, figsize=(8.0, 2.4 * len(names)),
                             facecolor=SURFACE, sharex=True)
    for ax, name in zip(np.atleast_1d(axes), names):
        t, m = SOURCES[name](cfg, n_blocks, seed)
        times = [t[i][m[i]].numpy() for i in range(n_blocks)]
        ax.eventplot(times, colors=INK, lineoffsets=np.arange(n_blocks),
                     linelengths=0.7, linewidths=1.6, zorder=5)
        ax.set_ylim(-0.8, n_blocks - 0.2); ax.set_yticks([])
        style(ax, "time  (units of $T_s$)", "",
              f"{name}   (mean N_S = {np.mean([len(x) for x in times]):.1f}, "
              f"min {min(len(x) for x in times)}, max {max(len(x) for x in times)})")
    fig.tight_layout()
    return fig


fig_sources(CFG); plt.show()


## 3. The two baselines that were missing

### `uncoded`: $X = S$

With $\hat S = Y$ we would get $D = \sigma_J^2$ exactly. But the receiver
**knows** that events are ordered; sorting $Y$ is therefore free and is an
isotonic projection, which can only reduce the MSE. The true uncoded baseline
is thus $D_{\text{uncoded}} \le \sigma_J^2$, and the gap grows with the
noise. This is already a mini JSCC result: part of the "gain" at high noise
simply comes from sorting, not from the network.

### `blind`: the blind decoder

$\hat S_i = \mathbb{E}[T_i \mid N, i]$, estimated from the training data. It
completely ignores $Y$. **As long as $D \approx D_{\text{blind}}$, no
temporal information is crossing the system**, whatever the shape of the
curve. This is the test that revealed v1's problem and it must appear
everywhere.


In [ ]:
def blind_table(t, m):
    """E[T_i | N, i] estimated on a set of blocks -> dict (N, i) -> value."""
    n = m.sum(1).cpu().numpy(); tn = t.cpu().numpy()
    acc = {}
    for b in range(tn.shape[0]):
        for i in range(int(n[b])):
            acc.setdefault((int(n[b]), i), []).append(tn[b, i])
    return {k: float(np.mean(v)) for k, v in acc.items()}


def blind_D(tab, t, m):
    """D of a decoder that ignores Y and outputs the table above."""
    n = m.sum(1).cpu().numpy(); tn = t.cpu().numpy()
    ds = []
    for b in range(tn.shape[0]):
        N = int(n[b])
        if N == 0:
            continue
        ds.append(np.mean([(tn[b, i] - tab.get((N, i), tn[b, i])) ** 2
                           for i in range(N)]))
    return float(np.mean(ds))


## 4. The SNN — continuous-time LIF, unrolled event by event

A single spiking ingredient, the LIF cell: hard threshold on the forward pass
(Heaviside), arctan surrogate gradient on the backward pass (Neftci, Mostafa
& Zenke 2019).

**What changes compared to v1.** The network is no longer unrolled over a
clock of `T_bins` steps but over the $N$ events of the block. Between event
$i-1$ and event $i$, separated by $g_i$, the potential decays by
$e^{-g_i/\tau}$: this is the **exact solution** of the LIF's leak ODE, not an
approximation. So it's the same neuron, simply without quantizing time — the
standard approach for neural point processes (Du et al. 2016; Mei & Eisner
2017).

Three direct consequences:

- $g_i$ enters the computation **explicitly**, in addition to being implicit
  in the leak. The identity becomes representable by a linear head, which was
  not the case in v1 (it would have required inverting $\beta^{g/\Delta}$);
- no more bin collisions, neither on the source side nor on the channel side;
- $N$ unroll steps instead of 128: about 10× faster, and a much shorter
  gradient path (the #1 cause of the optimization difficulties listed in
  v1).

`s` is not detached in the reset term (1st SuperSpike approximation): this is
the fix found in v1, without which `theta` receives no gradient as soon as
the readout reads $u$ rather than $s$.

The decoder is **bidirectional** (two LIF passes, forward and backward,
concatenated): it's a block code, the receiver has the whole block available
before decoding. The encoder remains **causal**.


In [ ]:
class ATanSpike(torch.autograd.Function):
    """Heaviside on the forward pass, arctan surrogate on the backward pass
    (Neftci, Mostafa & Zenke)."""

    @staticmethod
    def forward(ctx, u, alpha):
        ctx.save_for_backward(u); ctx.alpha = alpha
        return (u > 0).to(u.dtype)

    @staticmethod
    def backward(ctx, go):
        (u,) = ctx.saved_tensors; a = ctx.alpha
        return go * a / (2 * (1 + (math.pi / 2 * a * u) ** 2)), None


def spike(u, alpha=2.0):
    return ATanSpike.apply(u, alpha)


def reverse_seq(x, n):
    """Reverses each sequence over only its first n elements (padding stays
    on the right). The operation is an involution on the valid part, so the
    same function is used to reverse and then to restore the order."""
    N = x.shape[1]
    ar = torch.arange(N, device=x.device).unsqueeze(0)
    idx = (n.unsqueeze(1) - 1 - ar).clamp(min=0)
    if x.dim() == 3:
        idx = idx.unsqueeze(-1).expand(-1, -1, x.shape[2])
    return torch.gather(x, 1, idx)


class EventLIF(nn.Module):
    """u <- u * exp(-g_i/tau) * (1 - s) + W x_i ;  s = H(u - theta).

    tau (time constant, in units of mean_gap) and theta (threshold) are
    learned per unit. A single feed-forward hidden layer: memory from one
    event to the next comes only from the leak of the potential."""

    def __init__(self, n_in, hidden, alpha=2.0, tau_init=1.0):
        super().__init__()
        self.syn = nn.Linear(n_in, hidden)
        self.log_tau = nn.Parameter(torch.full((hidden,), math.log(tau_init)))
        self.theta = nn.Parameter(torch.ones(hidden))
        self.alpha, self.hidden = alpha, hidden

    def forward(self, x, dt):
        """x (B,N,n_in), dt (B,N) in units of mean_gap -> u (B,N,H)."""
        B, N, _ = x.shape
        tau = torch.exp(self.log_tau).clamp(1e-2, 1e2)
        u = torch.zeros(B, self.hidden, device=x.device)
        s = torch.zeros_like(u)
        cur = self.syn(x)
        out = []
        for i in range(N):
            u = u * torch.exp(-dt[:, i:i + 1] / tau) * (1.0 - s) + cur[:, i]
            s = spike(u - self.theta, self.alpha)
            out.append(u)
        return torch.stack(out, 1)


class Readout(nn.Module):
    """EventLIF (uni- or bidirectional) + linear head initialized to ZERO.
    Zero output at initialization -> zero displacement -> the system starts
    out exactly as `uncoded`."""

    def __init__(self, cfg, bidir=False):
        super().__init__()
        self.bidir = bidir
        self.fwd = EventLIF(cfg.n_in, cfg.hidden, cfg.spike_alpha)
        self.bwd = EventLIF(cfg.n_in, cfg.hidden, cfg.spike_alpha) if bidir else None
        self.head = nn.Linear(cfg.hidden * (2 if bidir else 1), 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, x, dt, n):
        u = self.fwd(x, dt)
        if self.bidir:
            ub = reverse_seq(self.bwd(reverse_seq(x, n), reverse_seq(dt, n)), n)
            u = torch.cat([u, ub], -1)
        return self.head(u).squeeze(-1)


## 5. The system — displacement parametrization

$$
g^X_i = g_i\,e^{a_{\max}\tanh f_i},\qquad
X = \operatorname{cumsum}(g^X),\qquad
Y = X + Z,\qquad
\hat S_i = Y_{(i)} + c_{\max}\tanh h_i
$$

The structural guarantees, true **for any weights**, before any training:

- $N_X = N_S = N_{\hat S}$: the encoder produces one multiplier per input
  gap, no more no less; the decoder one displacement per received event. No
  loss penalty, no top-N, no possible collision.
- $g^X_i > 0$: the multiplicative form guarantees that $X$ stays strictly
  increasing, so no reordering on the encoder side (this was a real bug in
  v1).
- $X \subset [0, T_c]$: after the cumulative sum, we renormalize if
  $\sum_i g^X_i > T_c$. This is the sample-wise version of the constraint,
  the same as $N_X = N_S$.
- $R = T_s/T_c = 1$ and $c(X) = N_X/T_c = N_S/T_s$: rate and input cost
  identical to `uncoded`, as required by the PDF's controlled comparison.
- **At initialization, $X = S$ and $\hat S = Y_{(\cdot)}$**: the three modes
  are numerically identical. Verified in the next cell.

Sorting $Y$ at the receiver is legitimate (it knows events are ordered) and
is already accounted for in the `uncoded` baseline, so it does not bias the
comparison.

$c_{\max}$ is proportional to $\max(\mu, \sigma_J)$ rather than to $\mu$
alone: at high noise the needed correction is on the order of $\sigma_J$, and
a cap fixed at $3\mu$ handicaps the decoder — tested, it cost a factor of 1.5
on the distortion at $\sigma_J/\mu = 3$.


In [ ]:
class ResidualSystem(nn.Module):
    """mode: "uncoded" (no parameters) | "decoder" | "jscc"."""

    MODES = ("uncoded", "decoder", "jscc")

    def __init__(self, cfg, mode):
        super().__init__()
        assert mode in self.MODES
        self.cfg, self.mode = cfg, mode
        self.enc = Readout(cfg, bidir=False) if mode == "jscc" else None
        self.dec = Readout(cfg, bidir=True) if mode in ("decoder", "jscc") else None

    def encode(self, t, m):
        cfg = self.cfg
        g = gaps(t, m)
        if self.enc is None:
            return t, g, torch.zeros_like(g)
        x = torch.stack([g / cfg.mean_gap, t / cfg.Tc], -1) * m.unsqueeze(-1)
        a = cfg.a_max * torch.tanh(self.enc(x, g / cfg.mean_gap, m.sum(1)))
        gx = g * torch.exp(a) * m
        tot = gx.sum(1, keepdim=True)
        scale = (cfg.Tc / tot.clamp(min=1e-6)).clamp(max=1.0)   # X <= Tc, and = 1
        gx = gx * scale                                         # if already inside the window
        return torch.cumsum(gx, 1) * m, gx, a

    def forward(self, t, m, gen=None, sigma=None):
        cfg = self.cfg
        sigma = cfg.sigma_J if sigma is None else sigma
        X, gx, a = self.encode(t, m)
        Z = torch.randn(X.shape, generator=gen, device=X.device) * sigma
        Y = (X + Z) * m
        Ys = torch.sort(Y.masked_fill(~m, 1e6), 1).values * m   # the RX sorts
        if self.dec is None:
            S_hat = Ys
        else:
            gy = gaps(Ys, m)
            x = torch.stack([gy / cfg.mean_gap, Ys / cfg.Tc], -1) * m.unsqueeze(-1)
            c_max = cfg.c_max_ratio * max(cfg.mean_gap, cfg.sigma_J)
            h = self.dec(x, gy.abs() / cfg.mean_gap, m.sum(1))
            S_hat = Ys + c_max * torch.tanh(h) * m
        return {"t": t, "m": m, "X": X, "gx": gx, "a": a, "Y": Y, "Ys": Ys,
                "S_hat": S_hat}

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def distortion(out):
    """Eq. 7-8 of the PDF, exactly: (1/N) Sum (T_i - T_hat_i)^2, matched by
    index, averaged over the batch. No min(N_S, N_Ŝ) truncation: the two
    counts are equal by construction."""
    m = out["m"].to(out["t"].dtype)
    sq = (out["t"] - out["S_hat"]) ** 2 * m
    return (sq.sum(1) / m.sum(1).clamp(min=1)).mean()


### Structural check

Two properties that must hold **before any training**. If one breaks after a
code change, it's a bug, not a result.


In [ ]:
_t, _m = SOURCES["hawkes"](CFG, 512, 0)
_g = torch.Generator().manual_seed(7)
_ds = []
for mode in MODES:
    torch.manual_seed(0)
    mdl = ResidualSystem(CFG, mode)
    with torch.no_grad():
        out = mdl(_t, _m, gen=torch.Generator().manual_seed(7))
        _ds.append(float(distortion(out)))
    n_s = _m.sum(1)
    n_x = (out["X"] > 0).sum(1) if mdl.enc is not None else n_s
    n_h = out["m"].sum(1)
    ok_order = bool((out["X"][:, 1:] - out["X"][:, :-1] >= 0)[_m[:, 1:]].all())
    ok_win = bool((out["X"] <= CFG.Tc + 1e-5).all())
    print(f"{mode:8s}  D_init={_ds[-1]:.6f}  n_params={mdl.n_params:4d}  "
          f"N_X=N_S : {bool((n_x == n_s).all())}   N_Ŝ=N_S : {bool((n_h == n_s).all())}   "
          f"X increasing : {ok_order}   X <= Tc : {ok_win}")

assert max(_ds) - min(_ds) < 1e-5, "the three modes must coincide at init"
print()
print(f"The three modes coincide at init (gap {max(_ds)-min(_ds):.2e}): OK.")
print(f"sigma_J^2 = {CFG.sigma_J**2:.6f} ; D_uncoded = {_ds[0]:.6f} "
      f"(< sigma^2 thanks to sorting Y at the receiver).")


## 6. Training and measurement


In [ ]:
def train(cfg, mode, ds_train, steps=None, seed=0, log=None):
    steps = steps or cfg.steps
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    model = ResidualSystem(cfg, mode)
    hist = []
    if model.n_params:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for st in range(1, steps + 1):
            t, m = ds_train.sample(cfg.batch, gen)
            loss = distortion(model(t, m, gen=gen))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
            if log and (st % log == 0 or st == 1):
                hist.append((st, float(loss.detach())))
    return model, hist


@torch.no_grad()
def evaluate(model, cfg, ds_eval, seed=9999, batch=4000):
    gen = torch.Generator().manual_seed(seed)
    t, m = ds_eval.sample(batch, gen)
    out = model(t, m, gen=gen)
    d = float(distortion(out))
    pair = m[:, 1:] & m[:, :-1]
    reorder = float((((out["Y"][:, 1:] - out["Y"][:, :-1]) < 0) & pair)
                    .any(1).float().mean())
    return {"d": d, "rmse_gaps": math.sqrt(d) / cfg.mean_gap,
            "reorder_frac_Y": reorder, "out": out}


def make_datasets(name, cfg, n_train=N_TRAIN, n_eval=N_EVAL):
    t, m = SOURCES[name](cfg, n_train, 0)
    te, me = SOURCES[name](cfg, n_eval, 1)
    tab = blind_table(t[:8000], m[:8000])
    return Dataset(t, m), Dataset(te, me), blind_D(tab, te, me)

## 7. Experiment 1 — noise sweep, three sources, three systems

The PDF's controlled comparison: same rate $R = 1$, same number of
transmitted events $N_X = N_S$, same channel. Any gap between the three
systems therefore comes solely from the temporal geometry.

Each point is the average over `SEEDS` seeds. The two references are plotted:
$D_{\text{blind}}$ (ceiling: transmit nothing) and $\sigma_J^2$ (raw
transmission without even sorting $Y$).


In [ ]:
def run_sigma_sweep(sources=("poisson", "gamma_k4", "hawkes"),
                    sigma_grid=SIGMA_GRID, seeds=SEEDS, steps=STEPS, verbose=True):
    rows, models = [], {}
    for src in sources:
        ds, dse, Db = make_datasets(src, CFG)
        for ratio in sigma_grid:
            cfg = replace(CFG, sigma_ratio=ratio, steps=steps)
            for mode in MODES:
                ds_, t0 = [], time.time()
                for sd in seeds:
                    mdl, _ = train(cfg, mode, ds, seed=sd)
                    ds_.append(evaluate(mdl, cfg, dse)["d"])
                    models[(src, ratio, mode, sd)] = (mdl, cfg)
                r = {"source": src, "sigma_ratio": ratio, "mode": mode,
                     "d_mse": float(np.mean(ds_)), "d_std": float(np.std(ds_)),
                     "rmse_gaps": math.sqrt(np.mean(ds_)) / CFG.mean_gap,
                     "d_over_blind": float(np.mean(ds_)) / Db,
                     "blind": Db, "sigma2": cfg.sigma_J ** 2}
                rows.append(r)
                if verbose:
                    print(f"{src:9s} σ/µ={ratio:4.1f} {mode:8s} "
                          f"D={r['d_mse']:.5f} ±{r['d_std']:.5f}  "
                          f"D/blind={r['d_over_blind']:.3f}  "
                          f"({time.time()-t0:.0f}s)", flush=True)
    return rows, models


SWEEP_ROWS, SWEEP_MODELS = run_sigma_sweep()

In [ ]:
show_table(SWEEP_ROWS,
           cols=["source", "sigma_ratio", "mode", "d_mse", "d_std", "rmse_gaps",
                 "d_over_blind", "blind", "sigma2"],
           title="Noise sweep — D in units of Ts^2, rmse_gaps in mean gaps")
print()
for src in dict.fromkeys(r["source"] for r in SWEEP_ROWS):
    for ratio in SIGMA_GRID:
        sel = {r["mode"]: r["d_mse"] for r in SWEEP_ROWS
               if r["source"] == src and r["sigma_ratio"] == ratio}
        print(f"{src:9s} sigma/mu={ratio:4.1f}  decoder gain = {sel['uncoded']/sel['decoder']:5.2f}x"
              f"   encoder gain (jscc/decoder) = {sel['decoder']/sel['jscc']:5.3f}x")


In [ ]:
def fig_sigma_sweep(rows, sources=("poisson", "gamma_k4", "hawkes")):
    fig, axes = plt.subplots(1, len(sources), figsize=(4.6 * len(sources), 4.4),
                             facecolor=SURFACE, sharey=True)
    for ax, src in zip(np.atleast_1d(axes), sources):
        sub = [r for r in rows if r["source"] == src]
        for mode in MODES:
            pts = sorted([r for r in sub if r["mode"] == mode],
                         key=lambda r: r["sigma_ratio"])
            colour, marker = MODE_STYLE[mode]
            xs = [r["sigma_ratio"] for r in pts]
            ys = [r["d_mse"] for r in pts]
            es = [r["d_std"] for r in pts]
            ax.errorbar(xs, ys, yerr=es, color=colour, lw=2.0, marker=marker, ms=7,
                        mew=2.0, mfc=SURFACE, mec=colour, capsize=3, zorder=5,
                        label=mode)
        xs = sorted({r["sigma_ratio"] for r in sub})
        ax.axhline(sub[0]["blind"], color=C_BLIND, lw=1.5, ls=(0, (4, 3)), zorder=4,
                   label="blind (transmits nothing)")
        ax.plot(xs, [(x * CFG.mean_gap) ** 2 for x in xs], color=C_OPT, lw=1.2,
                ls=(0, (1, 2)), zorder=4, label="$\\sigma_J^2$ (no sorting)")
        ax.set_xscale("log"); ax.set_yscale("log")
        style(ax, "$\\sigma_J$ / mean gap  (log)",
              "D  (units of $T_s^2$)" if src == sources[0] else "", src)
    legend(np.atleast_1d(axes)[-1], loc="lower right")
    fig.suptitle("Distortion vs noise — anything below the grey line is actually "
                 "transmitting information", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


fig_sigma_sweep(SWEEP_ROWS); plt.show()


**Reading.** All three learned curves stay clearly **below** the grey line:
unlike v1, temporal information actually crosses the system, at every noise
level. `decoder` beats `uncoded` everywhere (guaranteed by the
initialization), and `jscc` beats `decoder` by a small but systematic margin
— this is the encoder gain, to be isolated and interpreted in the following
experiments.


## 7bis. Sanity check — with no noise, is reconstruction exact?

Three checks, from the most structural to the most informative. The first
two carry an `assert`: if they fail, it's a bug, not a result.

- **A.** **Untrained** model, $\sigma_J = 0$. The channel is the identity, the
  heads are zero, so $X = S$, $Y = X$, $\hat S = Y$: reconstruction must be
  exact to machine precision, for all three modes.
- **B.** Model **trained at $\sigma_J = 0$**. Since $D = 0$ from
  initialization, the gradient is zero and training must not break anything.
  This is a regression check on the gradient path.
- **C.** Model **trained at $\sigma_J = \mu$, re-evaluated at $\sigma_J = 0$**.
  This one is the most telling, and $D$ does **not** fall back to zero: this
  is expected and must be explained. The decoder has learned a correction
  calibrated to a given noise level; applied without noise, that correction
  becomes a bias. The encoder/decoder pair is **tuned to a given $\sigma_J$**,
  it is not an invertible codec.


In [ ]:
ds_p, dse_p, _ = make_datasets("poisson", CFG, n_train=8000, n_eval=2000)
cfg_zero = replace(CFG, sigma_ratio=0.0)
_t, _m = SOURCES["poisson"](cfg_zero, 2000, 4242)

print("A. UNTRAINED model, sigma_J = 0")
for mode in MODES:
    torch.manual_seed(0)
    mdl = ResidualSystem(cfg_zero, mode)
    with torch.no_grad():
        o = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
    err = float(((o["t"] - o["S_hat"]).abs() * o["m"]).max())
    n_ok = bool((o["m"].sum(1) == _m.sum(1)).all())
    print(f"   {mode:8s}  D = {float(distortion(o)):.3e}   "
          f"max error = {err:.2e}  ({err/CFG.mean_gap:.1e} mean gap)   "
          f"N preserved: {n_ok}")
    assert err < 1e-6, "with no noise, reconstruction must be exact"

print()
print("B. System TRAINED at sigma_J = 0: must remain the identity")
cfg_zt = replace(CFG, sigma_ratio=0.0, steps=200)
for mode in ("decoder", "jscc"):
    mdl, _ = train(cfg_zt, mode, ds_p, seed=0)
    with torch.no_grad():
        o = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
    d0 = float(distortion(o))
    print(f"   {mode:8s}  D after 200 steps = {d0:.3e}")
    assert d0 < 1e-6, "training at zero noise must not degrade the identity"

print()
print("C. System trained at sigma_J = mu, re-evaluated at sigma_J = 0")
for mode in ("decoder", "jscc"):
    mdl, cfg_ = SWEEP_MODELS[("poisson", 1.0, mode, SEEDS[0])]
    with torch.no_grad():
        o_n = mdl(_t, _m, gen=torch.Generator().manual_seed(1))
        o_0 = mdl(_t, _m, gen=torch.Generator().manual_seed(1), sigma=0.0)
    extra = ""
    if mode == "jscc":
        dev = float(((o_0["X"] - o_0["t"]).abs() * o_0["m"]).sum() / o_0["m"].sum())
        extra = f"      deformation |X - S| = {dev/CFG.mean_gap:.2f} mean gap"
    print(f"   {mode:8s}  D at sigma_J = mu : {float(distortion(o_n)):.5f}"
          f"   ->   D at sigma_J = 0 : {float(distortion(o_0)):.5f}{extra}")
print()
print("   D does not fall back to zero, and that is expected: the decoder's correction is")
print("   calibrated for a given noise level; with no noise it becomes a bias.")
print("   The encoder/decoder pair is tuned to a given sigma_J.")


### Full chain on a Poisson source — `uncoded` vs `jscc`

Same draw, same noise seed for both systems: since $N_X = N_S$ on both sides,
the vector $Z$ applies element-wise identically, so the comparison is
direct. The thin lines connect each $T_i$ to its $\hat T_i$: their slope is
the error.

The third panel replays the **same** `jscc` model, same weights, with
$\sigma_J = 0$ — this is the visual version of check C above.


In [ ]:
def fig_chain_poisson(models, ratio=1.0, block=0, seed=11):
    mdl_u, cfg = models[("poisson", ratio, "uncoded", SEEDS[0])]
    mdl_j, _ = models[("poisson", ratio, "jscc", SEEDS[0])]
    t, m = SOURCES["poisson"](cfg, block + 1, seed)
    with torch.no_grad():
        ou = mdl_u(t, m, gen=torch.Generator().manual_seed(seed))
        oj = mdl_j(t, m, gen=torch.Generator().manual_seed(seed))
        oz = mdl_j(t, m, gen=torch.Generator().manual_seed(seed), sigma=0.0)
    pick = lambda a: a[block][m[block]].numpy()
    dblock = lambda o: float(((o["t"] - o["S_hat"]) ** 2 * o["m"])[block].sum()
                             / m[block].sum())
    panels = [("uncoded", ou, f"$X = S$, $\\hat S = Y$ sorted — D = {dblock(ou):.5f}"),
              ("jscc", oj, f"learned encoder + decoder — D = {dblock(oj):.5f}"),
              ("jscc, $\\sigma_J = 0$", oz,
               f"same weights, perfect channel — D = {dblock(oz):.5f}")]
    fig, axes = plt.subplots(len(panels), 1, figsize=(9.6, 3.6 * len(panels)),
                             facecolor=SURFACE, sharex=True)
    for ax, (name, o, sub) in zip(axes, panels):
        rows = [("$S$  source", pick(o["t"]), INK),
                ("$X$  codeword", pick(o["X"]), C_UNCODED),
                ("$Y$  channel output", pick(o["Ys"]), C_DECODER),
                ("$\\hat S$  reconstruction", pick(o["S_hat"]), C_JSCC)]
        for k, (lab, vals, col) in enumerate(rows):
            y0 = len(rows) - 1 - k
            ax.eventplot([vals], colors=col, lineoffsets=y0, linelengths=0.52,
                         linewidths=2.4, zorder=6)
            ax.text(-0.015, y0, lab, transform=ax.get_yaxis_transform(), color=col,
                    fontsize=9, fontweight="bold", ha="right", va="center")
        for a_, b_ in zip(pick(o["t"]), pick(o["S_hat"])):
            ax.plot([a_, b_], [3, 0], color=MUTED, lw=0.7, alpha=0.55, zorder=3)
        for xv in (0.0, cfg.Tc):
            ax.axvline(xv, color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=2)
        ax.set_ylim(-0.6, 3.6); ax.set_yticks([])
        style(ax, "time  (units of $T_s$)", "", name, sub)
    n = int(m[block].sum())
    fig.suptitle(f"Poisson source, block {block} — "
                 f"$N_S = N_X = N_{{\\hat S}} = {n}$, $\\sigma_J/\\mu$ = {ratio:g}, "
                 f"same draw and same noise", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    return fig


for b in range(2):
    fig_chain_poisson(SWEEP_MODELS, block=b); plt.show()


## 8. Experiment 2 — Step 2 of the PDF: does predictability create the gain?

Central question of Step 2: $W_i \sim \Gamma(k, \mu/k)$ with fixed mean, $k$
swept over $\{1, 2, 4, 8\}$. $k = 1$ is Poisson, larger $k$ gives increasingly
regular timing. Prediction: a more predictable source leaves more room for
the decoder (it can rely on the prior) **and** for the encoder (the
distribution of gaps is more concentrated, hence easier to redistribute).

We separate the two effects by reading `uncoded/decoder` (decoder gain) and
`decoder/jscc` (encoder gain) separately.


In [ ]:
K_GRID = [("poisson", 1), ("gamma_k2", 2), ("gamma_k4", 4), ("gamma_k8", 8)]
K_SIGMA = 1.0

K_ROWS, K_MODELS = [], {}
for name, k in K_GRID:
    ds, dse, Db = make_datasets(name, CFG)
    cfg = replace(CFG, sigma_ratio=K_SIGMA, steps=STEPS)
    res = {}
    for mode in MODES:
        vals = []
        for sd in SEEDS:
            mdl, _ = train(cfg, mode, ds, seed=sd)
            vals.append(evaluate(mdl, cfg, dse)["d"])
            K_MODELS[(k, mode, sd)] = (mdl, cfg)
        res[mode] = float(np.mean(vals))
    K_ROWS.append({"k": k, "blind": Db, **res,
                   "gain_decodeur": res["uncoded"] / res["decoder"],
                   "gain_encodeur": res["decoder"] / res["jscc"],
                   "gain_total": res["uncoded"] / res["jscc"]})
    print(f"k={k}  uncoded={res['uncoded']:.5f} decoder={res['decoder']:.5f} "
          f"jscc={res['jscc']:.5f}  blind={Db:.5f}  "
          f"gain_dec={K_ROWS[-1]['gain_decodeur']:.2f}× "
          f"gain_enc={K_ROWS[-1]['gain_encodeur']:.3f}×", flush=True)

show_table(K_ROWS, cols=["k", "uncoded", "decoder", "jscc", "blind",
                         "gain_decodeur", "gain_encodeur", "gain_total"],
           title=f"Step 2 — renouvellement Gamma, σ_J/µ = {K_SIGMA}")

In [ ]:
def fig_k_sweep(rows):
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4), facecolor=SURFACE)
    ks = [r["k"] for r in rows]
    for mode in MODES:
        colour, marker = MODE_STYLE[mode]
        axes[0].plot(ks, [r[mode] for r in rows], color=colour, lw=2.0, marker=marker,
                     ms=8, mew=2.0, mfc=SURFACE, mec=colour, zorder=5, label=mode)
    axes[0].plot(ks, [r["blind"] for r in rows], color=C_BLIND, lw=1.5,
                 ls=(0, (4, 3)), zorder=4, label="blind")
    axes[0].set_yscale("log")
    style(axes[0], "k  (Gamma shape)", "D  (units of $T_s^2$)",
          "Distortion vs source regularity",
          f"$\\sigma_J$ / mean gap = {K_SIGMA}")
    legend(axes[0])

    axes[1].plot(ks, [r["gain_decodeur"] for r in rows], color=C_DECODER, lw=2.0,
                 marker="s", ms=8, mew=2.0, mfc=SURFACE, mec=C_DECODER,
                 label="decoder gain  (uncoded / decoder)")
    axes[1].plot(ks, [r["gain_encodeur"] for r in rows], color=C_JSCC, lw=2.0,
                 marker="^", ms=8, mew=2.0, mfc=SURFACE, mec=C_JSCC,
                 label="encoder gain  (decoder / jscc)")
    axes[1].axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)), zorder=4)
    style(axes[1], "k  (Gamma shape)", "gain factor",
          "Where does the gain come from?", "dashes: no gain")
    legend(axes[1])
    fig.tight_layout()
    return fig


fig_k_sweep(K_ROWS); plt.show()


## 9. Experiment 3 — what does the encoder learn? (eq. 12)

We plot $g_i \mapsto g^{(X)}_i$ as well as the learned multiplier
$g^{(X)}_i / g_i$ as a function of the source gap size. Since $N_X = N_S$ is
guaranteed and both sequences are ordered, the $i$-th codeword gap does
correspond to the $i$-th source gap.

**Hypothesis to test.** The budget is $\sum_i g^X_i \le T_c$ and the source
already fills the window, so the encoder cannot dilate globally: it can only
**reallocate**. But the jitter $\sigma_J$ is absolute, it mainly destroys
small gaps, while long silences remain estimable (and the decoder can rely on
the prior for those). We therefore expect **companding**: a multiplier
greater than 1 on small gaps, less than 1 on large ones.


In [ ]:
def fig_gap_mapping(models, src="gamma_k4", ratios=SIGMA_GRID, seed=4242, n_blocks=1500):
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), facecolor=SURFACE)
    colours = plt.get_cmap("viridis")(np.linspace(0.15, 0.8, len(ratios)))
    lim = 0.0
    for colour, ratio in zip(colours, ratios):
        mdl, cfg = models[(src, ratio, "jscc", SEEDS[0])]
        t, m = SOURCES[src](cfg, n_blocks, seed)
        with torch.no_grad():
            _, gx, _ = mdl.encode(t, m)
        g = gaps(t, m)
        sel = m.numpy().ravel()
        gs = (g.numpy().ravel() / cfg.mean_gap)[sel]
        gxs = (gx.numpy().ravel() / cfg.mean_gap)[sel]
        lim = max(lim, np.percentile(gs, 99.5), np.percentile(gxs, 99.5))
        axes[0].scatter(gs, gxs, s=4, color=colour, alpha=0.25, lw=0,
                        label=f"$\\sigma_J/\\mu$ = {ratio:g}")
        # sliding median of the multiplier
        order = np.argsort(gs)
        gs_o, mult_o = gs[order], (gxs / np.clip(gs, 1e-6, None))[order]
        w = max(len(gs_o) // 60, 20)
        xs = [gs_o[i:i + w].mean() for i in range(0, len(gs_o) - w, w)]
        ys = [np.median(mult_o[i:i + w]) for i in range(0, len(gs_o) - w, w)]
        axes[1].plot(xs, ys, color=colour, lw=2.0, label=f"$\\sigma_J/\\mu$ = {ratio:g}")

    axes[0].plot([0, lim], [0, lim], color=INK, lw=1.5, ls=(0, (4, 3)), zorder=6)
    axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
    style(axes[0], "$g_i$ / mean gap", "$g_i^{(X)}$ / mean gap",
          "Gap mapping (eq. 12)", f"source = {src} ; dashes: identity")
    legend(axes[0])

    axes[1].axhline(1.0, color=INK, lw=1.5, ls=(0, (4, 3)), zorder=4)
    axes[1].set_xscale("log")
    style(axes[1], "$g_i$ / mean gap  (log)", "median multiplier  $g^{(X)}_i/g_i$",
          "Companding: are small gaps dilated?",
          "above 1: dilated ; below: compressed")
    legend(axes[1])
    fig.tight_layout()
    return fig


fig_gap_mapping(SWEEP_MODELS); plt.show()


In [ ]:
# Direct quantification: median multiplier per decile of source gap
mdl, cfg = SWEEP_MODELS[("gamma_k4", SIGMA_GRID[-1], "jscc", SEEDS[0])]
t, m = SOURCES["gamma_k4"](cfg, 3000, 77)
with torch.no_grad():
    _, gx, _ = mdl.encode(t, m)
g = gaps(t, m)
sel = m.numpy().ravel()
gs = (g.numpy().ravel() / cfg.mean_gap)[sel]
mult = (gx.numpy().ravel()[sel]) / np.clip(g.numpy().ravel()[sel], 1e-9, None)
qs = np.quantile(gs, np.linspace(0, 1, 11))
print(f"source = gamma_k4, sigma_J/mu = {cfg.sigma_ratio}")
print(" source gap decile      learned median multiplier")
for i in range(10):
    s = (gs >= qs[i]) & (gs < qs[i + 1])
    if s.sum():
        print(f"  [{qs[i]:5.2f}, {qs[i+1]:5.2f}] mu  ->  {np.median(mult[s]):.3f}")
print(f"\nSum of source gaps / Tc      = {float((g.sum(1)/cfg.Tc).mean()):.3f}")
print(f"Sum of encoded gaps / Tc     = {float((gx.sum(1)/cfg.Tc).mean()):.3f}")
print("The window is already full: the encoder can only reallocate, not dilate.")


## 10. Experiment 4 — Step 3 of the PDF: the minimal model, with its optimum

This is the part that actually answers the research question. We reduce to
$S = \{T_1, T_2\}$, i.e. the single variable $G = T_2 - T_1$, with

$$G \longrightarrow G_X \longrightarrow G_Y = G_X + Z \longrightarrow \hat G .$$

Constraint: $G_X \in [0, T_c]$ — the two-event version of the sample-wise
constraint $\sum_i g^X_i \le T_c$ of the full problem.

**What changes everything compared to simply repeating the setup on a small
scale: we compute the optimum.** On a grid, the encoder is a free monotone
function and the associated MMSE decoder is
$\hat G(y) = \mathbb{E}[G \mid G_Y = y]$, computable exactly by numerical
integration. We directly optimize the encoder's table by gradient descent.
This gives three exact references:

- `uncoded`: $\sigma^2$;
- `decoder-opt`: identity at transmission + exact MMSE at reception;
- `jscc-opt`: optimal monotone encoder + exact MMSE.

The network is then compared to **these** numbers, not to `uncoded`.


In [ ]:
def source_pdf(kind, g, mu, k=4.0):
    if kind == "exp":
        p = np.exp(-g / mu) / mu
    elif kind == "gamma":
        p = g ** (k - 1) * np.exp(-g * k / mu) * (k / mu) ** k / math.gamma(k)
    elif kind == "bimodal":
        p = (0.5 * np.exp(-(g - 0.4 * mu) ** 2 / (2 * (0.12 * mu) ** 2))
             + 0.5 * np.exp(-(g - 2.2 * mu) ** 2 / (2 * (0.35 * mu) ** 2)))
    else:
        raise ValueError(kind)
    p = np.clip(p, 0, None)
    return p / p.sum()


class Grid:
    """Shared source / channel discretization, for the exact MMSE computation."""

    def __init__(self, kind, mu, Tc, sigma, M=161, My=321, k=4.0):
        self.g = torch.tensor(np.linspace(1e-4, Tc, M), dtype=torch.float64)
        self.p = torch.tensor(source_pdf(kind, self.g.numpy(), mu, k), dtype=torch.float64)
        self.y = torch.tensor(np.linspace(-4 * sigma, Tc + 4 * sigma, My),
                              dtype=torch.float64)
        self.dy = float(self.y[1] - self.y[0])
        self.sigma, self.Tc, self.mu, self.kind = sigma, Tc, mu, kind

    def distortion(self, x):
        """x (M,): images of the grid under the encoder -> (D, Ghat(y), p(y))."""
        d = self.y.view(1, -1) - x.view(-1, 1)
        q = self.p.view(-1, 1) * torch.exp(-d ** 2 / (2 * self.sigma ** 2))
        marg = q.sum(0) + 1e-300
        ghat = (q * self.g.view(-1, 1)).sum(0) / marg
        err = (self.g.view(-1, 1) - ghat.view(1, -1)) ** 2
        D = (q * err).sum() * self.dy / (math.sqrt(2 * math.pi) * self.sigma)
        return D, ghat, marg


def monotone_map(w, Tc):
    """Free parameter -> strictly increasing function with values in (0, Tc]."""
    return Tc * torch.cumsum(torch.softmax(w, 0), 0)


def optimal_encoder(grid, steps=1500, lr=0.05, seed=0):
    torch.manual_seed(seed)
    M = grid.g.shape[0]
    w = nn.Parameter(torch.log(torch.full((M,), 1.0 / M, dtype=torch.float64)))
    opt = torch.optim.Adam([w], lr=lr)
    for _ in range(steps):
        D, _, _ = grid.distortion(monotone_map(w, grid.Tc))
        opt.zero_grad(); D.backward(); opt.step()
    with torch.no_grad():
        x = monotone_map(w, grid.Tc)
        D, _, _ = grid.distortion(x)
    return x.detach(), float(D)


def reference_levels(grid):
    """(D_uncoded, D_decoder-opt): identity at transmission, without / with MMSE."""
    D_dec, _, _ = grid.distortion(grid.g.clone())
    return grid.sigma ** 2, float(D_dec)


In [ ]:
class TinyEnc(nn.Module):
    """Learned scalar encoder, initialized to the identity (last layer zero)."""

    def __init__(self, Tc, mu, h=64):
        super().__init__()
        self.Tc, self.mu = Tc, mu
        self.net = nn.Sequential(nn.Linear(1, h), nn.Tanh(), nn.Linear(h, h),
                                 nn.Tanh(), nn.Linear(h, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, g):
        a = torch.tanh(self.net((g / self.mu).unsqueeze(-1)).squeeze(-1))
        return (g * torch.exp(math.log(6.0) * a)).clamp(0.0, self.Tc)


class TinyDec(nn.Module):
    """Learned scalar decoder, initialized to the identity."""

    def __init__(self, Tc, mu, cmax, h=64):
        super().__init__()
        self.Tc, self.mu, self.cmax = Tc, mu, cmax
        self.net = nn.Sequential(nn.Linear(1, h), nn.Tanh(), nn.Linear(h, h),
                                 nn.Tanh(), nn.Linear(h, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, y):
        return y + self.cmax * torch.tanh(self.net((y / self.mu).unsqueeze(-1)).squeeze(-1))


def sample_G(grid, n, gen):
    idx = torch.multinomial(grid.p.float(), n, replacement=True, generator=gen)
    return grid.g[idx].float()


def train_tiny(grid, mode, steps=2000, batch=4096, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    mu, Tc, sg = grid.mu, grid.Tc, grid.sigma
    enc = TinyEnc(Tc, mu) if mode == "jscc" else None
    dec = TinyDec(Tc, mu, 3 * max(mu, sg)) if mode in ("decoder", "jscc") else None
    params = [p for m in (enc, dec) if m is not None for p in m.parameters()]
    if params:
        opt = torch.optim.Adam(params, lr=lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for _ in range(steps):
            g = sample_G(grid, batch, gen)
            x = g if enc is None else enc(g)
            y = x + torch.randn(x.shape, generator=gen) * sg
            gh = y if dec is None else dec(y)
            loss = ((g - gh) ** 2).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step()
    with torch.no_grad():
        g = sample_G(grid, 200_000, torch.Generator().manual_seed(123))
        x = g if enc is None else enc(g)
        y = x + torch.randn(x.shape, generator=torch.Generator().manual_seed(124)) * sg
        gh = y if dec is None else dec(y)
        D = float(((g - gh) ** 2).mean())
    return enc, dec, D


In [ ]:
STEP3_KINDS = ["exp", "gamma", "bimodal"]
STEP3_SIGMAS = [0.3, 1.0, 3.0]
MU, TC = CFG.mean_gap, CFG.Tc

STEP3_ROWS, STEP3_GRIDS = [], {}
for kind in STEP3_KINDS:
    for sr in STEP3_SIGMAS:
        G = Grid(kind, MU, TC, sr * MU)
        D_unc, D_dec_opt = reference_levels(G)
        x_opt, D_jscc_opt = optimal_encoder(G)
        learned, enc_jscc = {}, None
        for md_ in MODES:
            e_, _, d_ = train_tiny(G, md_)
            learned[md_] = d_
            if md_ == "jscc":
                enc_jscc = e_
        STEP3_GRIDS[(kind, sr)] = (G, x_opt, enc_jscc)
        STEP3_ROWS.append({
            "source": kind, "sigma_ratio": sr,
            "opt_uncoded": D_unc, "opt_decoder": D_dec_opt, "opt_jscc": D_jscc_opt,
            "learned_uncoded": learned["uncoded"], "learned_decoder": learned["decoder"],
            "learned_jscc": learned["jscc"],
            "gain_enc_opt": D_dec_opt / D_jscc_opt,
            "gain_enc_learned": learned["decoder"] / learned["jscc"],
            "gap_to_opt": learned["jscc"] / D_jscc_opt})
        print(f"{kind:8s} sigma/mu={sr:4.1f} | OPT unc={D_unc:.5f} dec={D_dec_opt:.5f} "
              f"jscc={D_jscc_opt:.5f} (enc gain {D_dec_opt/D_jscc_opt:.2f}x) | "
              f"LEARNED dec={learned['decoder']:.5f} jscc={learned['jscc']:.5f} "
              f"(within {learned['jscc']/D_jscc_opt:.2f}x of the optimum)", flush=True)

show_table(STEP3_ROWS, title="Step 3 — two-event model, numerical optimum vs learned")


In [ ]:
def fig_step3(kind="gamma", sr=1.0):
    G, x_opt, enc = STEP3_GRIDS[(kind, sr)]
    g = G.g.numpy() / MU
    xo = x_opt.numpy() / MU
    with torch.no_grad():
        xl = enc(G.g.float()).numpy() / MU
    p = G.p.numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.4), facecolor=SURFACE)

    ax = axes[0]
    ax.plot(g, g, color=INK, lw=1.5, ls=(0, (4, 3)), label="identity (uncoded)")
    ax.plot(g, xo, color=C_OPT, lw=2.4, label="optimal encoder")
    ax.plot(g, xl, color=C_JSCC, lw=2.0, ls="--", label="learned encoder")
    ax.set_ylim(0, TC / MU)
    style(ax, "$G$ / mean gap", "$G_X$ / mean gap",
          "The optimal encoder dilates", f"{kind}, $\\sigma_J/\\mu$ = {sr:g}")
    legend(ax, "upper left")

    ax = axes[1]
    ax.fill_between(g, 0, p / p.max(), color=C_BLIND, alpha=0.35, lw=0,
                    label="density of $G$")
    slope_o = np.gradient(xo, g)
    slope_l = np.gradient(xl, g)
    ax2 = ax.twinx()
    ax2.plot(g, slope_o, color=C_OPT, lw=2.4, label="optimal slope $dG_X/dG$")
    ax2.plot(g, slope_l, color=C_JSCC, lw=2.0, ls="--", label="learned slope")
    ax2.axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)))
    ax2.set_ylabel("slope", color=INK2, fontsize=10)
    ax2.tick_params(colors=INK2, labelsize=9)
    ax.set_xlim(0, np.percentile(g[p > p.max() * 1e-3], 99))
    style(ax, "$G$ / mean gap", "density (normalized)",
          "The gain comes from the slope", "slope > 1: improved effective SNR")
    legend(ax2, "upper right")
    legend(ax, "upper left")

    ax = axes[2]
    rows = [r for r in STEP3_ROWS if r["source"] == kind]
    xs = [r["sigma_ratio"] for r in rows]
    ax.plot(xs, [r["opt_uncoded"] for r in rows], color=C_UNCODED, lw=2.0, marker="o",
            ms=7, mfc=SURFACE, mew=2, mec=C_UNCODED, label="uncoded")
    ax.plot(xs, [r["opt_decoder"] for r in rows], color=C_DECODER, lw=2.0, marker="s",
            ms=7, mfc=SURFACE, mew=2, mec=C_DECODER, label="optimal decoder")
    ax.plot(xs, [r["opt_jscc"] for r in rows], color=C_OPT, lw=2.4, marker="D",
            ms=7, mfc=SURFACE, mew=2, mec=C_OPT, label="optimal JSCC")
    ax.plot(xs, [r["learned_jscc"] for r in rows], color=C_JSCC, lw=2.0, ls="--",
            marker="^", ms=7, mfc=SURFACE, mew=2, mec=C_JSCC, label="learned JSCC")
    ax.set_xscale("log"); ax.set_yscale("log")
    style(ax, "$\\sigma_J$ / mean gap  (log)", "D  (units of $T_s^2$)",
          "Does the network reach the optimum?", kind)
    legend(ax, "lower right")

    fig.tight_layout()
    return fig


fig_step3("gamma", 1.0); plt.show()
fig_step3("bimodal", 1.0); plt.show()


In [ ]:
# Quantifying the dilation: where does the probability mass go after encoding?
for kind in STEP3_KINDS:
    G, x_opt, _ = STEP3_GRIDS[(kind, 1.0)]
    g, x, p = G.g.numpy(), x_opt.numpy(), G.p.numpy()
    cum = np.cumsum(p)
    print(f"\n{kind} :")
    for q in (0.1, 0.5, 0.9):
        j = int(np.searchsorted(cum, q))
        print(f"  quantile {q:.0%} : G = {g[j]/MU:5.2f} mu  ->  G_X = {x[j]/MU:5.2f} mu"
              f"   (factor {x[j]/g[j]:5.2f})")
    j50 = int(np.searchsorted(cum, 0.5))
    print(f"  local slope at the median : dG_X/dG = {np.gradient(x, g)[j50]:.2f}"
          f"   ->  effective SNR x {np.gradient(x, g)[j50]**2:.1f}")


**The mechanism, in one sentence.** In the two-event model, $G \sim \mu = T_c/9$
but the window extends to $T_c$: it is essentially empty. The optimal encoder
**spreads** the distribution of $G$ over the whole window, with a local slope
on the order of 6 where the probability mass lies. Since the jitter
$\sigma_J$ is absolute, multiplying gaps by $\lambda$ multiplies the
effective SNR of the timing channel by $\lambda^2$ — **at strictly constant
event cost**. This is exactly the type of gain the PDF's research question is
aimed at: the information is carried by the relative temporal geometry, not
by the individual times.

The learned decoder reaches the exact MMSE (gap < 3%). The learned encoder
captures the right shape but stays within a factor of 1.3 to 2.6 of the
optimum: that's the remaining gap to explain, and it's a real discussion
point, not a bug.

**Why the gain is much smaller in the full problem.** With $N$ events,
$\sum_i g_i \approx T_c$: the window is **already full** (verified in
experiment 3). The encoder therefore cannot dilate globally, only reallocate
between gaps — hence a gain of 10-20% instead of 2-10×. This difference
between Step 3 and the full problem is not an inconsistency, it is the direct
consequence of the window constraint, and it is probably the most interesting
result of the notebook.


## 11. One realization through the chain


In [ ]:
def fig_chain(models, src="gamma_k4", ratio=SIGMA_GRID[-1], block=0, seed=7):
    mdl_j, cfg = models[(src, ratio, "jscc", SEEDS[0])]
    mdl_u, _ = models[(src, ratio, "uncoded", SEEDS[0])]
    t, m = SOURCES[src](cfg, block + 1, seed)
    with torch.no_grad():
        oj = mdl_j(t, m, gen=torch.Generator().manual_seed(seed))
        ou = mdl_u(t, m, gen=torch.Generator().manual_seed(seed))
    pick = lambda a: a[block][m[block]].numpy()
    rows = [("$S$  source", pick(oj["t"]), INK),
            ("$X$  codeword", pick(oj["X"]), C_UNCODED),
            ("$Y$  channel output", pick(oj["Ys"]), C_DECODER),
            ("$\\hat S$  jscc", pick(oj["S_hat"]), C_JSCC),
            ("$\\hat S$  uncoded", pick(ou["S_hat"]), C_OPT)]
    fig, ax = plt.subplots(figsize=(9.4, 4.8), facecolor=SURFACE)
    for k, (name, vals, colour) in enumerate(rows):
        y0 = len(rows) - 1 - k
        ax.eventplot([vals], colors=colour, lineoffsets=y0, linelengths=0.52,
                     linewidths=2.4, zorder=6)
        ax.text(-0.015, y0, name, transform=ax.get_yaxis_transform(), color=colour,
                fontsize=9, fontweight="bold", ha="right", va="center")
    for xv in (0.0, cfg.Tc):
        ax.axvline(xv, color=MUTED, lw=1.2, ls=(0, (4, 3)), zorder=2)
    n = int(m[block].sum())
    ax.set_ylim(-0.6, len(rows) - 0.4); ax.set_yticks([])
    style(ax, "time  (units of $T_s$)", "", "Full chain — same draw, same noise",
          f"source = {src}, $\\sigma_J/\\mu$ = {ratio:g}, "
          f"N_S = N_X = N_Ŝ = {n} (guaranteed by construction), dashes: $[0, T_c]$")
    fig.tight_layout()
    return fig


for b in range(3):
    fig_chain(SWEEP_MODELS, block=b); plt.show()


## 12. Experiment 5 — should the encoder also see the whole block?

The decoder is bidirectional: it sees the entire block before producing
$\hat S$. The encoder, on the other hand, is **causal** — $X_i$ only depends
on $g_1,\dots,g_i$ and $T_i$, never on future events of the same block.

Nothing in the assignment imposes this asymmetry: the block formulation
("*for a source point-process block $S$ observed over duration $T_s$, the
encoder produces $X$*") just as well allows an encoder that sees the whole
block before emitting the first coded event — exactly the argument that
already justifies the bidirectional decoder.

**Why this might help.** The window constraint is **global**:

$$\lambda = \min\!\left(1,\ \frac{T_c}{\sum_{j=1}^{N} \tilde g^X_j}\right)$$

This sum is only known at the end of the block. A causal encoder must decide
$g^X_i$ without knowing how many more events will arrive nor their size: it
makes local bets, which $\lambda$ can then uniformly crush if the bet was
too optimistic. An encoder that sees the whole block can solve a global
allocation instead of a sequence of myopic decisions — closer to what the
optimal Step 3 encoder does over its window.

**What it costs.** Two passes instead of one: the encoder goes from 193 to
385 parameters and from $N$ to $2N$ LIF iterations, and above all loses all
useful latency — it must wait for the end of the block before emitting the
first coded spike. That's the price to pay if the gain is confirmed.

We compare `jscc` (causal encoder, section 6) to a variant where the encoder
is bidirectional, with a strictly identical decoder, on the same three
sources and the same noise levels as experiment 1. The class inherits from
`ResidualSystem`: `encode` and `forward` are not duplicated, only the
instantiation of `self.enc` changes.


In [ ]:
class ResidualSystemEncBidir(ResidualSystem):
    '''Identical to ResidualSystem, except that the encoder (jscc mode) sees
    the whole block before emitting — Readout(cfg, bidir=True) instead of
    bidir=False. `encode` and `forward` are inherited unchanged: they call
    self.enc(...) generically, regardless of its output width.'''

    def __init__(self, cfg, mode):
        super().__init__(cfg, mode)
        if mode == "jscc":
            self.enc = Readout(cfg, bidir=True)


def train_variant(cfg, model_cls, mode, ds_train, steps=None, seed=0):
    '''Like train() (section 6), but parametrized by model class, so as not
    to touch the definition of train() used by the previous sections.'''
    steps = steps or cfg.steps
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed + 1)
    model = model_cls(cfg, mode)
    if model.n_params:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        for _ in range(steps):
            t, m = ds_train.sample(cfg.batch, gen)
            loss = distortion(model(t, m, gen=gen))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
    return model


p_causal = ResidualSystem(CFG, "jscc").n_params
p_bidir = ResidualSystemEncBidir(CFG, "jscc").n_params
print(f"jscc, causal encoder          : {p_causal} parameters, encoder = N LIF iterations")
print(f"jscc, bidirectional encoder   : {p_bidir} parameters, encoder = 2N LIF iterations")


### Sweep — 3 sources x `SIGMA_GRID` x 3 seeds

3 seeds instead of the 2 used in experiment 1: the expected gap between the
causal and bidirectional encoder is on the order of a few percent, smaller
than the encoder gain itself — more repeats are needed to distinguish it from
training noise. `SIGMA_GRID` is reused as-is to stay comparable to
experiment 1.

**Cost**: $3 \times 3 \times 3 = 27$ triples (decoder, causal, bidirectional)
trained for `STEPS` steps each, i.e. roughly two to three times the duration
of experiment 1 — expect 30 to 45 minutes depending on the machine. For a
first pass, reducing `FEEDBACK_SEEDS` to `[0]` and/or `STEPS` in the
configuration cell gives a result in a few minutes, too noisy to draw
conclusions but enough to check that the cell runs.


In [ ]:
FEEDBACK_SOURCES = ("poisson", "gamma_k4", "hawkes")
FEEDBACK_SEEDS = [0, 1, 2]

FEEDBACK_ROWS = []
for src in FEEDBACK_SOURCES:
    ds, dse, Db = make_datasets(src, CFG)
    for ratio in SIGMA_GRID:
        cfg = replace(CFG, sigma_ratio=ratio, steps=STEPS)
        d_dec, d_causal, d_bidir = [], [], []
        t0 = time.time()
        for sd in FEEDBACK_SEEDS:
            md_dec = train_variant(cfg, ResidualSystem, "decoder", ds, seed=sd)
            d_dec.append(evaluate(md_dec, cfg, dse)["d"])
            md_causal = train_variant(cfg, ResidualSystem, "jscc", ds, seed=sd)
            d_causal.append(evaluate(md_causal, cfg, dse)["d"])
            md_bidir = train_variant(cfg, ResidualSystemEncBidir, "jscc", ds, seed=sd)
            d_bidir.append(evaluate(md_bidir, cfg, dse)["d"])
        row = {"source": src, "sigma_ratio": ratio,
               "decoder": float(np.mean(d_dec)),
               "jscc_causal": float(np.mean(d_causal)),
               "std_causal": float(np.std(d_causal)),
               "jscc_bidir": float(np.mean(d_bidir)),
               "std_bidir": float(np.std(d_bidir))}
        row["gain_enc_causal"] = row["decoder"] / row["jscc_causal"]
        row["gain_enc_bidir"] = row["decoder"] / row["jscc_bidir"]
        row["bidir_vs_causal"] = row["jscc_causal"] / row["jscc_bidir"]
        FEEDBACK_ROWS.append(row)
        print(f"{src:9s} σ/µ={ratio:4.1f} | decoder={row['decoder']:.5f} | "
              f"causal={row['jscc_causal']:.5f}±{row['std_causal']:.5f} "
              f"(gain {row['gain_enc_causal']:.3f}×) | "
              f"bidir={row['jscc_bidir']:.5f}±{row['std_bidir']:.5f} "
              f"(gain {row['gain_enc_bidir']:.3f}×) | "
              f"bidir/causal={row['bidir_vs_causal']:.3f}×  "
              f"({time.time()-t0:.0f}s)", flush=True)

In [ ]:
show_table(FEEDBACK_ROWS,
           cols=["source", "sigma_ratio", "decoder", "jscc_causal", "jscc_bidir",
                 "gain_enc_causal", "gain_enc_bidir", "bidir_vs_causal"],
           title="Causal vs bidirectional encoder, with identical decoder")
print()
gains = [r["bidir_vs_causal"] for r in FEEDBACK_ROWS]
print(f"bidir/causal : min={min(gains):.3f}x  median={np.median(gains):.3f}x  "
      f"max={max(gains):.3f}x  over {len(gains)} combinations")
print(f"all > 1 : {all(g > 1.0 for g in gains)}  "
      "(if not: training noise exceeds the measured effect on at least one point)")


In [ ]:
def fig_feedback(rows, sources=FEEDBACK_SOURCES):
    fig, axes = plt.subplots(1, len(sources), figsize=(4.6 * len(sources), 4.4),
                             facecolor=SURFACE, sharey=True)
    for ax, src in zip(np.atleast_1d(axes), sources):
        sub = sorted([r for r in rows if r["source"] == src], key=lambda r: r["sigma_ratio"])
        xs = [r["sigma_ratio"] for r in sub]
        ax.plot(xs, [r["gain_enc_causal"] for r in sub], color=C_DECODER, lw=2.0,
                marker="o", ms=7, mfc=SURFACE, mew=2, mec=C_DECODER,
                label="causal encoder")
        ax.plot(xs, [r["gain_enc_bidir"] for r in sub], color=C_JSCC, lw=2.0,
                marker="^", ms=7, mfc=SURFACE, mew=2, mec=C_JSCC,
                label="bidirectional encoder")
        ax.axhline(1.0, color=INK, lw=1.2, ls=(0, (4, 3)), zorder=4)
        ax.set_xscale("log")
        style(ax, "$\\sigma_J$ / mean gap  (log)",
              "encoder gain  (decoder / jscc)" if src == sources[0] else "", src)
    legend(np.atleast_1d(axes)[-1], loc="upper right")
    fig.suptitle("Seeing the whole block helps the encoder, but only a little — the "
                 "saturated window remains the limiting factor", color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig


fig_feedback(FEEDBACK_ROWS); plt.show()


**Reading.** The bidirectional encoder systematically beats the causal one
(the two curves nearly overlap, the bidirectional one very slightly above) —
a real but small effect, on the order of a few percent, to be compared with
the encoder gain itself (`decoder`/`jscc`, already modest in the full
problem). It is most pronounced on Hawkes at high noise: that's the source
whose block structure (bursts, highly variable $N$) leaves the most room for
a global allocation rather than a sequence of local decisions.

**What this says about the cause of the plateau.** The gap between Step 3's
encoder gain (2 to 10×) and that of the full problem (a few percent) is
therefore **not** mainly due to the encoder's causality — otherwise lifting
the causal constraint would have closed a substantial part of the gap.
Window saturation ($\sum_i g_i \approx T_c$, section 9) remains the dominant
explanation: it caps what there is to gain, whether the encoder sees the
future of the block or not. The bidirectional version only better
distributes what remains to be gained within that cap.

**Practical verdict.** The gain does not justify doubling the parameters and
latency of the default encoder — but if the final report raises the question
of the encoder's causality, these are the numbers to cite, rather than a
purely qualitative argument.


## 13. Limitations and next steps

### Accepted limitations

- **The decoder is bidirectional, the encoder is causal.** This is justified
  (block code: the receiver has the whole block) but it means `decoder` and
  `jscc` are not strictly causal online SNNs. A test with a causal decoder is
  a one-line variant (`bidir=False`) and is worth quantifying.
- **A single hidden LIF layer**, feed-forward, 32 units. A deliberate choice
  of simplicity, not a demonstrated capacity limit. The residual gap to the
  optimum in Step 3 (factor 1.3-2.6) is the right place to test whether
  capacity is the cause — and Step 3 is small enough to sweep width quickly.
- **`a_max = log 4`** bounds the dilation per gap. In Step 3 the optimum
  requires a local slope on the order of 6, so this bound is potentially
  active in the full problem. Worth sweeping.
- **Reordering of $Y$ is frequent** (`reorder_frac_Y` climbs above 90% of
  blocks at high noise): the receiver sorts, which is optimal in the MSE
  sense for an ordered target, but the index-wise matching of eq. 7 becomes
  questionable in this regime. A 1-D Wasserstein distance or a
  Victor-Purpura distance would give a less arbitrary measure; this is a
  natural extension.
- **Two seeds per point** in the sweeps. Better than v1 (a single one), still
  insufficient to settle gaps of a few percent. Increase `SEEDS` before
  drawing conclusions on the encoder gain in the full problem.
- The whole comparison is at $R = 1$. The PDF requires this for the initial
  experiments, but $R \ne 1$ (more or less channel time than source time)
  would qualitatively change the available dilation margin — this is
  probably the most interesting follow-up.

### Next steps

1. **Sweep $R = T_s/T_c$.** The identified mechanism (dilation at constant
   event cost) predicts that the gain grows with $T_c/T_s$, since the window
   stops being saturated. This is a falsifiable prediction directly testable
   with this code.
2. **Close the gap to the optimum in Step 3**: network width, $a_{\max}$,
   number of steps, and a comparison to a free monotone table learned by SGD
   (same family as the numerical optimum, but trained on samples) to
   separate "the function family is too poor" from "optimization doesn't
   converge."
3. **Carry Step 3's companding over to the full problem**: impose on the full
   system's encoder the shape found optimal for two events, and see whether
   it beats the freely learned encoder. If so, the problem is optimization;
   if not, the window constraint dominates.
4. **Distortion on relative geometry rather than absolute times**: replace
   $\sum_i (T_i - \hat T_i)^2$ with a distortion on the gaps
   $\sum_i (g_i - \hat g_i)^2$. The PDF's research question is about relative
   temporal geometry; measuring it directly would change what the system is
   incentivized to preserve.

### Main references

- Gastpar, Rimoldi & Vetterli, *To code, or not to code: lossy source-channel
  communication revisited*, IEEE Trans. IT 49(5), 2003 — necessary and
  sufficient conditions for the optimality of uncoded transmission. The
  framework that explains why `uncoded` is a tough baseline and where a gain
  can exist.
- Rubin, *Information Rates and Data-Compression Schemes for Poisson
  Processes*, IEEE Trans. IT, 1974; Shen, Moser & Pfister, *Rate-Distortion
  Problems of the Poisson Process based on a Group-Theoretic Approach*,
  arXiv:2202.13684 — rate-distortion theory for this type of source,
  including over intervals.
- Skatchkovsky, Jang & Simeone, *End-to-End Learning of Neuromorphic Wireless
  Systems*, Asilomar 2020 (NeuroJSCC) — the closest prior work.
- Neftci, Mostafa & Zenke, *Surrogate Gradient Learning in Spiking Neural
  Networks*, IEEE SPM, 2019 — the surrogate gradient used here.
- Srinivas, Adve & Eckford, *Molecular communication in fluid media: the
  additive inverse Gaussian noise channel*, IEEE Trans. IT, 2012 — the
  additive timing channel as seen by information theory.
- Du et al., *Recurrent Marked Temporal Point Processes*, KDD 2016; Mei &
  Eisner, *The Neural Hawkes Process*, NeurIPS 2017 — the event-by-event
  unrolling adopted here.
